# 模块一：数据获取与准备
本笔记本用于准备基础分析数据，包括两部分：
1. **上市公司财报数据**：下载上市旅游公司营收数据（需要网络环境，基于 `akshare`）。
2. **景区 AOI**：转换百度地图解析出来的网页边界点坐标至 WGS84 面状（Polygon）以供后续地图匹配使用。

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import akshare as ak
import os
import glob
from config import STOCK_FINANCE_DIR, AOI_DIR

## 1. 抓取上市公司财报原始数据

In [ ]:
# 需要分析的股票池
stock_lst = ["sh600054","sz000888","sz000430","sh603099"]

concat_df = pd.DataFrame([])

for stock_symbol in stock_lst:
    # 获取利润表数据
    try:
        profit_sheet_df = ak.stock_profit_sheet_by_report_em(symbol=stock_symbol)
        revenue_data = profit_sheet_df[['SECUCODE', 'SECURITY_CODE', 'SECURITY_NAME_ABBR',
                                        'REPORT_DATE_NAME', 'REPORT_DATE', 
                                        'TOTAL_OPERATE_INCOME', 'OPERATE_INCOME']]
        concat_df = pd.concat([concat_df, revenue_data], ignore_index=True)
    except Exception as e:
        print(f"获取 {stock_symbol} 失败: {e}")

if not concat_df.empty:
    # 简单清洗并设定年份
    concat_df['REPORT_DATE'] = pd.to_datetime(concat_df['REPORT_DATE'])
    concat_df['YEAR'] = concat_df['REPORT_DATE'].dt.year
    
    # 取 2009-2025 年之间
    concat_df = concat_df[(concat_df['YEAR'] >= 2009) & (concat_df['YEAR'] <= 2025)].reset_index(drop=True)
    
    print("抓取到的公司名单:", concat_df['SECURITY_NAME_ABBR'].unique())
    
    # 保存
    finance_csv_path = os.path.join(STOCK_FINANCE_DIR, "营收_4家公司.csv")
    concat_df.to_csv(finance_csv_path, index=False)
    print(f"财报数据已保存至 {finance_csv_path}")

## 2. 处理百度地图景区边界(AOI)
百度地图网页抓取下来的文本是 `lng,lat,lng,lat...` 这种平铺格式，以下演示如何转换并生成 `shp` 文件。

> 注：坐标系转换需使用外部脚本完成，此处只演示从最终干净的 wgs84 csv 转为 shp 面模型

In [ ]:
# 假设在 AOI_DIR 下已经有经过脱敏和清理的 wgs84 坐标 CSV
wgs84_files = glob.glob(os.path.join(AOI_DIR, "*wgs84.csv"))

if not wgs84_files:
    raise FileNotFoundError(f"未在 {AOI_DIR} 找到包含 wgs84 的 AOI 坐标 CSV 文件。")

for csv_file in wgs84_files:
    # 获取景区名称（文件名去掉无用后缀）
    aoi_name = os.path.basename(csv_file).split("_")[0] 
    
    wgs84_df = pd.read_csv(csv_file)
    if 'lng' in wgs84_df.columns and 'lat' in wgs84_df.columns:
        # 构多边形
        polygon = Polygon(zip(wgs84_df['lng'], wgs84_df['lat']))
        gdf = gpd.GeoDataFrame(index=[0], crs='EPSG:4326', geometry=[polygon])
        
        out_shp = os.path.join(AOI_DIR, f"百度地图AOI_{aoi_name}.shp")
        gdf.to_file(out_shp, encoding='utf-8')
        print(f"成功生成 AOI Shapefile: {out_shp}")
    else:
        print(f"跳过 {csv_file}，无法识别 lng 和 lat 字段")

print("预处理全部完成。")